In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import os

In [2]:
dataset_path = "dataset/dataset-resized"

In [3]:
img_size = (128, 128)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2   # 80% train, 20% validation
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.


In [4]:
model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

C:\Users\O\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [6]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 20s 269ms/step - accuracy: 0.3641 - loss: 1.4875 - val_accuracy: 0.4076 - val_loss: 1.5204
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 19s 242ms/step - accuracy: 0.5415 - loss: 1.2073 - val_accuracy: 0.4235 - val_loss: 1.3902
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 15s 240ms/step - accuracy: 0.6018 - loss: 1.0723 - val_accuracy: 0.4473 - val_loss: 1.3731
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 21s 247ms/step - accuracy: 0.6467 - loss: 0.9325 - val_accuracy: 0.5149 - val_loss: 1.3234
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 16s 244ms/step - accuracy: 0.6907 - loss: 0.8139 - val_accuracy: 0.4453 - val_loss: 1.5023


In [7]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Compression ON
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("waste_model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\O\AppData\Local\Temp\tmp0f5q1a85\assets


INFO:tensorflow:Assets written to: C:\Users\O\AppData\Local\Temp\tmp0f5q1a85\assets


Saved artifact at 'C:\Users\O\AppData\Local\Temp\tmp0f5q1a85'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1627647050576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625759137424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625759136848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625759137616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625761825232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625761824848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625759134544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625759137040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625761825808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1625761826768: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [8]:
size = os.path.getsize("waste_model.tflite") / (1024 * 1024)
print(f"Model size: {size:.2f} MB")

Model size: 0.80 MB
